In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\shikh\OneDrive\Desktop\AI_Banking_system\Data\01_support_tickets.csv")

In [3]:
df.head()

,ticket_id,date_created,customer_id,channel,category,sub_category,query_text,sentiment,risk_level,resolution_text,resolution_time_minutes,resolved_by,customer_satisfaction,escalated
0,TKT-10001,2023-03-13,CUST872246,Web Chat,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,I see a transaction of ₹999 I didn't make on 2...,Urgent,High,Unauthorized transaction reported on account. ...,28,Sneha R.,2,Yes
1,TKT-10002,2023-11-05,CUST127824,Branch,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,I see a transaction of ₹500 I didn't make on 1...,Neutral,High,Unauthorized transaction reported on account. ...,27,Auto-Resolved,5,No
2,TKT-10003,2023-08-05,CUST456778,Phone,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,"I see a transaction of ₹25,000 I didn't make o...",Neutral,High,Unauthorized transaction reported on account. ...,11,Anjali K.,2,No
3,TKT-10004,2023-01-23,CUST865179,Email,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,I see a transaction of ₹999 I didn't make on 1...,Confused,High,Unauthorized transaction reported on account. ...,8,Vikram P.,2,Yes
4,TKT-10005,2023-12-05,CUST338968,Phone,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,"I see a transaction of ₹25,000 I didn't make o...",Frustrated,High,Unauthorized transaction reported on account. ...,12,Priya S.,5,No


In [4]:
def map_risk_category(risk_level):
    if risk_level == "High":
        return "Fraud"
    elif risk_level == "Medium":
        return "Sensitive"
    elif risk_level == "Low":
        return "General"
    else:
        return "Unknown"

In [5]:
df["risk_category"] = df["risk_level"].apply(
    map_risk_category
)

In [6]:
df.head()

,ticket_id,date_created,customer_id,channel,category,sub_category,query_text,sentiment,risk_level,resolution_text,resolution_time_minutes,resolved_by,customer_satisfaction,escalated,risk_category
0,TKT-10001,2023-03-13,CUST872246,Web Chat,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,I see a transaction of ₹999 I didn't make on 2...,Urgent,High,Unauthorized transaction reported on account. ...,28,Sneha R.,2,Yes,Fraud
1,TKT-10002,2023-11-05,CUST127824,Branch,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,I see a transaction of ₹500 I didn't make on 1...,Neutral,High,Unauthorized transaction reported on account. ...,27,Auto-Resolved,5,No,Fraud
2,TKT-10003,2023-08-05,CUST456778,Phone,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,"I see a transaction of ₹25,000 I didn't make o...",Neutral,High,Unauthorized transaction reported on account. ...,11,Anjali K.,2,No,Fraud
3,TKT-10004,2023-01-23,CUST865179,Email,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,I see a transaction of ₹999 I didn't make on 1...,Confused,High,Unauthorized transaction reported on account. ...,8,Vikram P.,2,Yes,Fraud
4,TKT-10005,2023-12-05,CUST338968,Phone,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,"I see a transaction of ₹25,000 I didn't make o...",Frustrated,High,Unauthorized transaction reported on account. ...,12,Priya S.,5,No,Fraud


In [7]:
risk_df = df[['query_text','risk_category']]

In [8]:
risk_df.sample(10)

,query_text,risk_category
64,Why was my loan application rejected?,General
118,I updated my address but KYC is still showing ...,General
11,I received an OTP but never requested it,Fraud
175,My fixed deposit matured but amount not credited,General
41,I never signed up for this subscription but ₹2...,General
157,My account is showing negative balance but I d...,General
155,My account is showing negative balance but I d...,General
182,I want to add my wife as a joint account holder,General
34,Someone used my credit card online without my ...,Fraud
89,My loan top-up request has been pending for 2 ...,Sensitive


In [9]:
df2 = pd.read_csv(r"C:\Users\shikh\OneDrive\Desktop\AI_Banking_system\Data\risk_category_dataset_300.csv")

In [10]:
df3 = pd.read_csv(r"C:\Users\shikh\OneDrive\Desktop\AI_Banking_system\Data\risk_category_synthetic_1500_unique.csv")

In [11]:
risk_df = pd.concat([risk_df, df2,df3], ignore_index=True)

In [12]:
risk_df.head()

,query_text,risk_category
0,I see a transaction of ₹999 I didn't make on 2...,Fraud
1,I see a transaction of ₹500 I didn't make on 1...,Fraud
2,"I see a transaction of ₹25,000 I didn't make o...",Fraud
3,I see a transaction of ₹999 I didn't make on 1...,Fraud
4,"I see a transaction of ₹25,000 I didn't make o...",Fraud


In [13]:
risk_df.shape

(2000, 2)

In [14]:
import re
import pandas as pd

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Contractions ko normalize karo
    contractions = {
        "didn't": "did not",
        "can't": "can not",
        "couldn't": "could not",
        "won't": "will not",
        "wouldn't": "would not",
        "isn't": "is not",
        "wasn't": "was not",
        "aren't": "are not",
        "don't": "do not",
        "doesn't": "does not",
        "haven't": "have not",
        "hasn't": "has not"
    }

    for contraction, replacement in contractions.items():
        text = text.replace(contraction, replacement)

    # URLs remove
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Email remove
    text = re.sub(r'\S+@\S+', ' ', text)

    # Currency amounts remove
    # ₹999, ₹25,000, Rs 500, INR 1000 etc.
    text = re.sub(r'₹\s?[\d,]+', ' ', text)
    text = re.sub(r'\b(rs|inr)\.?\s?[\d,]+\b', ' ', text)

    # Remaining numbers remove
    text = re.sub(r'\d+', ' ', text)

    # Special characters remove
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Extra spaces remove
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [15]:
risk_df["cleaned_query"] = risk_df["query_text"].apply(clean_text)

In [16]:
risk_df[['cleaned_query','risk_category']].duplicated().sum()

np.int64(393)

In [17]:
risk_df= risk_df[['cleaned_query','risk_category']]

In [18]:
risk_df = risk_df.drop_duplicates()

In [19]:
risk_df.duplicated().sum()

np.int64(0)

In [20]:
risk_df.shape

(1607, 2)

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Features and target
X = risk_df["cleaned_query"]
y = risk_df["risk_category"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF
risk_tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_tfidf = risk_tfidf.fit_transform(X_train)
X_test_tfidf = risk_tfidf.transform(X_test)

# Logistic Regression with class imbalance handling
risk_model = LogisticRegression(
    max_iter=2000,
    C=2.0,
    class_weight="balanced",
    random_state=42
)

# Train
risk_model.fit(X_train_tfidf, y_train)

# Prediction
y_pred = risk_model.predict(X_test_tfidf)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9751552795031055

Classification Report:
              precision    recall  f1-score   support

       Fraud       0.95      0.99      0.97       107
     General       1.00      0.94      0.97       108
   Sensitive       0.97      0.99      0.98       107

    accuracy                           0.98       322
   macro avg       0.98      0.98      0.98       322
weighted avg       0.98      0.98      0.98       322



Testing

In [23]:
test_queries = [
    # Fraud
    "I noticed a card purchase that I never authorized",
    "A stranger appears to have made a payment from my account",
    "There is a UPI transfer in my history that I did not initiate",
    "My account was charged for an online order I never placed",
    "I think somebody has accessed my card information",
    "An unknown person withdrew cash using my account",
    "I received a payment alert but I did not make that payment",
    "There are suspicious charges appearing on my credit card",
    "Money has been sent from my account without my approval",
    "I want to report a transaction that looks fraudulent",

    # Sensitive
    "My identity documents are waiting for bank verification",
    "I need to submit my PAN for account verification",
    "Why has my KYC review not been completed yet",
    "I want to know what paperwork is needed for my home loan",
    "My personal loan is currently being reviewed",
    "The bank has asked me for additional KYC documents",
    "I need to correct information submitted during verification",
    "My loan documents have been submitted but are still under review",
    "My Aadhaar verification is not complete",
    "Please explain the requirements for updating my KYC",

    # General
    "I forgot the password for my online banking",
    "How can I see the balance of my savings account",
    "My ATM is not allowing me to withdraw cash",
    "I would like to download last month's statement",
    "How do I activate a newly received debit card",
    "My bank transfer has not appeared in my account yet",
    "How can I change my registered phone number",
    "Where can I find my previous transactions",
    "I need to request a fresh cheque book",
    "How do I change my internet banking PIN"
]

In [24]:
X_new = risk_tfidf.transform(test_queries)

predictions = risk_model.predict(X_new)

for query, prediction in zip(test_queries, predictions):
    print(f"{query} --> {prediction}")

I noticed a card purchase that I never authorized --> Fraud
A stranger appears to have made a payment from my account --> Fraud
There is a UPI transfer in my history that I did not initiate --> Fraud
My account was charged for an online order I never placed --> Fraud
I think somebody has accessed my card information --> Fraud
An unknown person withdrew cash using my account --> Fraud
I received a payment alert but I did not make that payment --> Fraud
There are suspicious charges appearing on my credit card --> Fraud
Money has been sent from my account without my approval --> Fraud
I want to report a transaction that looks fraudulent --> Fraud
My identity documents are waiting for bank verification --> Sensitive
I need to submit my PAN for account verification --> Sensitive
Why has my KYC review not been completed yet --> Sensitive
I want to know what paperwork is needed for my home loan --> Sensitive
My personal loan is currently being reviewed --> Sensitive
The bank has asked me for 

In [25]:
import joblib
import os

# models folder create karo
os.makedirs("models", exist_ok=True)

# Risk classification model save
joblib.dump(
    risk_model,
    "models/risk_category_model.pkl"
)

# TF-IDF vectorizer save
joblib.dump(
    risk_tfidf,
    "models/risk_category_tfidf.pkl"
)

print("Risk model saved successfully!")
print("TF-IDF vectorizer saved successfully!")

Risk model saved successfully!
TF-IDF vectorizer saved successfully!
